In [6]:
import torch
import torch.nn as nn
import math

In [7]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(x.size(-1))
        weights = torch.softmax(scores, dim=-1)

        return torch.matmul(weights, V)

In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = SelfAttention(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Linear(d_model, d_model)

    def forward(self, x):
        x = self.norm(x + self.attention(x))  # attention + residual
        x = self.norm(x + self.ff(x))         # feedforward + residual
        return x

In [9]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = TransformerBlock(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x)

In [10]:
vocab_size = 1000
model = SimpleTransformer(vocab_size, d_model=64)

x = torch.randint(0, vocab_size, (2, 10))
output = model(x)

print(output.shape)

torch.Size([2, 10, 1000])
